In [ ]:
import sys, os
os.chdir(os.path.abspath('..'))
sys.path.insert(0, os.getcwd())

In [ ]:
def get_tip_depth(tree, tip_name):
    # Find the clade matching the string name
    clade = next(tree.find_clades(target=tip_name), None)
    # Return the distance from root to this specific clade
    return tree.distance(tree.root, clade)

In [ ]:
from utils.utils import paths, load_pickle
from preprocessing.lepid.phylo import build_tree_lepid
from preprocessing.nymph.phylo import build_tree_nymph
from preprocessing.cub.phylo import build_tree_cub

from Bio import Phylo


tree_raw_bryo = Phylo.read(paths["raw_tree"]["bryo"], "newick")
tree_raw_cub = Phylo.read(paths["raw_tree"]["cub"], "newick")
tree_raw_lepid = Phylo.read(paths["raw_tree"]["lepid"], "newick")
tree_raw_nymph = Phylo.read(paths["raw_tree"]["nymph"], "newick")

# tree_cub = build_tree_cub()
# tree_lepid = build_tree_lepid()
# tree_nymph = build_tree_nymph()

tree_bryo = load_pickle(paths["metadata"]["bryo"] / "tree.pkl")
tree_cub = load_pickle(paths["metadata"]["cub"] / "tree.pkl")
tree_lepid = load_pickle(paths["metadata"]["lepid"] / "tree.pkl")
tree_nymph = load_pickle(paths["metadata"]["nymph"] / "tree.pkl")

cids_lepid = [tip.name for tip in tree_lepid.get_terminals()]
cids_nymph = [tip.name for tip in tree_nymph.get_terminals()]

class_data_lepid = load_pickle(paths["metadata"]["lepid"] / "class_data.pkl")

genera_lepid = set([cid.split("_")[0] for cid in cids_lepid])
genera_nymph = set([cid.split("_")[0] for cid in cids_nymph])

In [ ]:
tree_raw_bryo_names = [tip.name for tip in tree_raw_bryo.get_terminals()]
tree_raw_cub_names = [tip.name for tip in tree_raw_cub.get_terminals()]
tree_raw_lepid_names = [tip.name for tip in tree_raw_lepid.get_terminals()]
tree_raw_nymph_names = [tip.name for tip in tree_raw_nymph.get_terminals()]

bryo_raw_depths = [get_tip_depth(tree_raw_bryo, tip_name) for tip_name in tree_raw_bryo_names]
cub_raw_depths = [get_tip_depth(tree_raw_cub, tip_name) for tip_name in tree_raw_cub_names]
lepid_raw_depths = [get_tip_depth(tree_raw_lepid, tip_name) for tip_name in tree_raw_lepid_names]
nymph_raw_depths = [get_tip_depth(tree_raw_nymph, tip_name) for tip_name in tree_raw_nymph_names]

print(min(bryo_raw_depths), max(bryo_raw_depths))
print(min(cub_raw_depths), max(cub_raw_depths))
print(min(lepid_raw_depths), max(lepid_raw_depths))
print(min(nymph_raw_depths), max(nymph_raw_depths))

In [ ]:
tree_bryo_names = [tip.name for tip in tree_bryo.get_terminals()]
tree_cub_names = [tip.name for tip in tree_cub.get_terminals()]
tree_lepid_names = [tip.name for tip in tree_lepid.get_terminals()]
tree_nymph_names = [tip.name for tip in tree_nymph.get_terminals()]

bryo_depths = [get_tip_depth(tree_bryo, tip_name) for tip_name in tree_bryo_names]
cub_depths = [get_tip_depth(tree_cub, tip_name) for tip_name in tree_cub_names]
lepid_depths = [get_tip_depth(tree_lepid, tip_name) for tip_name in tree_lepid_names]
nymph_depths = [get_tip_depth(tree_nymph, tip_name) for tip_name in tree_nymph_names]

print(min(bryo_depths), max(bryo_depths))
print(min(cub_depths), max(cub_depths))
print(min(lepid_depths), max(lepid_depths))
print(min(nymph_depths), max(nymph_depths))

In [ ]:
from itertools import combinations

from utils.phylo import get_tree
from preprocessing.lepid.phylo import build_tree_lepid
from preprocessing.nymph.phylo import build_tree_nymph
from utils.utils import paths, load_pickle
from preprocessing.lepid.phylo import combine_trees_lepid_nymph
from preprocessing.common.phylo import prune_tree, augment_class_data

In [ ]:
def print_pairwise_distances(tree, labels):
    for a, b in combinations(labels, 2):
        print(f"tree.distance('{a}', '{b}') = {tree.distance(a, b)}")

def cids_starting_with(cids, prefix):
    return sorted([cid for cid in cids if cid.startswith(prefix)])

def labels_in_cids(cids, labels, header=None):
    if header:
        print(header)
    for cid in labels:
        print(cid in cids)

def cids_in_rank(cid, rank, class_data):
    rank_name = class_data[cid][rank]
    return sorted([cid for cid in class_data.keys() if class_data[cid][rank] == rank_name])

def neighbor_genus_dists(cid, tree, class_data, remove_genus_cid=True):

    cids_tree = {tip.name for tip in tree.get_terminals()}

    cd_cid = class_data[cid]
    rank = "family"
    if cd_cid["subfamily"] is not None:
        rank = "subfamily"
    if cd_cid["tribe"] is not None:
        rank = "tribe"

    print("Lowestest rank above genus:", rank)

    cids_cands = set(cids_in_rank(cid, rank, class_data)) & cids_tree
    if remove_genus_cid:
        cids_cands = {cid for cid in cids_cands if class_data[cid]["genus"] != cd_cid["genus"]}

    for cid_cand in sorted(cids_cands):
        print(f"{cid_cand}: {tree.distance(cid, cid_cand)}")

def closest_species_outside_genus(cid, tree, class_data):

    cids_tree = {tip.name for tip in tree.get_terminals()}

    cd_cid = class_data[cid]
    rank = "family"
    if cd_cid["subfamily"] is not None:
        rank = "subfamily"
    if cd_cid["tribe"] is not None:
        rank = "tribe"

    print("Lowestest rank above genus:", rank)

    cids_cands = set(cids_in_rank(cid, rank, class_data)) & cids_tree
    cids_cands = {cid for cid in cids_cands if class_data[cid]["genus"] != cd_cid["genus"]}

    cid_closest = None
    dist_closest = float("inf")
    for cid_cand in sorted(cids_cands):
        dist = tree.distance(cid, cid_cand)
        if dist < dist_closest:
            cid_closest = cid_cand
            dist_closest = dist

    print(f"Closest species outside genus: {cid_closest} (distance {dist_closest:.2f})")

In [ ]:
tree = get_tree(dataset="lepid")  # merged
cids_tree = {tip.name for tip in tree.get_terminals()}

tree_lepid = build_tree_lepid()
tree_nymph = build_tree_nymph()
cids_lepid = {tip.name for tip in tree_lepid.get_terminals()}
cids_nymph = {tip.name for tip in tree_nymph.get_terminals()}

class_data = load_pickle(paths["metadata"]["lepid"] / "class_data.pkl")
class_data_aug = augment_class_data(class_data, tree)

### closest genus to "colias" genus: "zerene"

In [ ]:
neighbor_genus_dists('colias_palaeno', tree, class_data_aug, remove_genus_cid=False)

### cid-combo distances

In [ ]:
labels = ['adelpha_mesentina', 'aglais_urticae', 'aglais_io', 'daedalma_dinias']
print_pairwise_distances(tree, labels)